In [ ]:
# author: Luka Kowalchuk
# student nr: 13326015
# year: 2025-2026

# This notebook was used to train and evaluate a Vision Transformer (ViT) to rapidly classify forensically relevant insects.
# The ViT forms a part of the Research Project:
# A Comparative Evaluation of Vision Transformer and Convolutional Neural Network Performance for Forensically Relevant Dipterans Classification
# as a part of the MSc Forensic Science at the University of Amsterdam, collaborating with the Wildlife Forensic Academy in South Africa
# The model is pretrained on BioCLIP: Samuel Stevens et al. “BioCLIP: A Vision Foundation Model for the Tree of Life”. In: Proceedings
# of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR). June 2024, pp. 19412–19424.
# Low Rank Adaptation Parameter Efficient Finetuning is applied using a dataset developed by the Wildlife Forensic Academy
# to represent forensically relevant dipterans species (adults and maggots) in South Africa
# The model was ran in Google Colab, hence the cells are designed for this purpose, but it can be ran locally with minor changes
# Claude Sonnet 4.6 was used to create this notebook, which was monitored and modified consistenly using unit-checks and other tests

# Make sure to check Cell 4 before running everything

In [ ]:
# =============================================================================
# CELL 0 — Install dependencies
# =============================================================================

# NOTE: torch, torchvision, Pillow, and numpy are pre-installed in Google Colab.
# If running locally, install PyTorch manually first for GPU support:
# https://pytorch.org/get-started/locally/

!pip install open_clip_torch peft scikit-learn matplotlib tqdm umap-learn plotly torchvision Pillow numpy torchao opencv-python-headless -q
!pip install -q --upgrade torchao

In [ ]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================
import os
import json
import random
import pickle
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import math
import cv2
import umap
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip as oc
from collections import defaultdict

from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from itertools import combinations

from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from peft import get_peft_model, LoraConfig

import plotly.graph_objects as go

from google.colab import drive

print("All imports successful.")

In [ ]:
# =============================================================================
# CELL 2 — Mount Google Drive & unzip dataset
# =============================================================================

drive.mount('/content/drive')

# Unzipping the dataset to local storage can improve preprocessing and training speed.
# Uncomment and update the path below to match your dataset location.
# !unzip /content/drive/MyDrive/final_dataset.zip -d /content/

In [ ]:
# =============================================================================
# CELL 3 — CONFIG
# =============================================================================
# This is the only cell you need to edit between runs. Make sure the
# augmentation directory is empty before
# =============================================================================

# --- MODE ---
# Choose 'flies' or 'maggots' to select which subset to train on.
MODE = 'flies'

# --- VERSION ---
# Choose 'BioCLIP-1' or 'BioCLIP-2'
VERSION = 'BioCLIP-1'


# --- PATHS ---
DATASET_PATH = '/content/final_dataset'                                         # path to unzipped dataset
AUG_DIR      = f'/content/drive/MyDrive/augmented_dataset_{MODE}'              # where augmented images are saved
CHECKPOINT   = f'/content/drive/MyDrive/{VERSION}_lora_best_{MODE}_test.pt'         # where best model is saved
PLOT_DIR     = f'/content/drive/MyDrive/{VERSION}_plots_{MODE}_test'                # where plots are saved
EMBED_DIR    = f'/content/drive/MyDrive/{VERSION}_embeddings_{MODE}_test'           # where embeddings are saved
EXPORT_DIR = f'/content/drive/MyDrive/{VERSION}_streamlit_export_{MODE}_test'  # where Streamlit app files are saved

os.makedirs(PLOT_DIR,  exist_ok=True)
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(AUG_DIR,   exist_ok=True)

# --- AUGMENTATIONS ---
# Add or remove augmentations as needed. Each entry is a function that takes a PIL image and returns a PIL image.
AUGMENTATIONS = [
    # lambda img: TF.rotate(img, 90),
    # lambda img: TF.hflip(TF.rotate(img, 180)),
    # lambda img: apply_blue_tint(img, intensity=BLUE_TINT_INTENSITY),
    # lambda img: apply_cutout(img, cutout_size=CUTOUT_SIZE),
]

BLUE_TINT_INTENSITY = 0.6    # only used if apply_blue_tint is uncommented above
CUTOUT_SIZE         = 128    # only used if apply_cutout is uncommented above

# --- TRAINING HYPERPARAMETERS ---
BATCH_SIZE        = 16
EPOCHS            = 10
LEARNING_RATE     = 1e-4
WEIGHT_DECAY      = 0.01
TRIPLET_MARGIN    = 0.2
K = 5  # number of neighbours for KNN classification and distance threshold analysis

# --- LORA HYPERPARAMETERS ---
LORA_RANK         = 8
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.1

# --- DATASET PARAMETERS ---
MIN_IMAGES             = 282    # minimum original images required for a class to be included
TRIAL_IMAGES_PER_CLASS = 250    # number of originals sampled per class for the trial dataset

# --- THRESHOLDS ---
THRESHOLDS = {
    ("BioCLIP-1", "flies"):   0.2110,
    ("BioCLIP-1", "maggots"): 0.2355,
    ("BioCLIP-2", "flies"):   0.1124,
    ("BioCLIP-2", "maggots"): 0.1184,
}

# --- HEATMAP ---
HEATMAP_N_PER_CLASS = 1    # number of attention rollout heatmaps saved per class

print(f"Config loaded — VERSION: {VERSION} | MODE: {MODE}")

In [ ]:
# =============================================================================
# CELL 4 — SafeImageFolder
# =============================================================================

class SafeImageFolder(datasets.ImageFolder):
    """
    Extends torchvision's ImageFolder to filter out corrupt images on load.
    Set verify=False to skip the scan (e.g. for already-validated datasets).
    """
    def __init__(self, root, transform=None, verify=True):
        super().__init__(root, transform=transform)

        if verify:
            original_count = len(self.samples)
            valid_samples  = []

            for path, label in self.samples:
                try:
                    img = Image.open(path)
                    img.verify()
                    valid_samples.append((path, label))
                except (UnidentifiedImageError, OSError, Exception):
                    print(f"  Removing corrupt image: {path}")

            self.samples = valid_samples
            self.imgs    = valid_samples
            removed      = original_count - len(valid_samples)
            print(f"  Dataset ready: {len(valid_samples)} valid images ({removed} corrupt removed)")

    def __getitem__(self, index):
        path, label  = self.samples[index]
        sample       = self.loader(path)
        if self.transform is not None:
            sample = self.transform(sample)
        return sample, label

In [ ]:
# =============================================================================
# CELL 5 — Load BioCLIP model
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

if VERSION == 'BioCLIP-1':
    model, _, preprocess = oc.create_model_and_transforms('hf-hub:imageomics/bioclip')
    distance_threshold = THRESHOLDS[(VERSION, MODE)]
elif VERSION == 'BioCLIP-2':
    model, _, preprocess = oc.create_model_and_transforms('hf-hub:imageomics/bioclip-2')
    distance_threshold = THRESHOLDS[(VERSION, MODE)]
else:
    print('Model not recognized')

model = model.to(device)
model.eval()
print(f"{VERSION} loaded.")

In [ ]:
# =============================================================================
# CELL 6 — Load dataset
# =============================================================================

dataset     = SafeImageFolder(DATASET_PATH, transform=preprocess)
class_names = sorted(dataset.classes)
num_classes = len(class_names)

print(f"Classes found: {num_classes}")
print(f"Total images:  {len(dataset)}")

In [ ]:
# =============================================================================
# CELL 7 — Data augmentation
# =============================================================================
# Generates augmented versions per image based on AUGMENTATIONS defined in CONFIG.
# Only processes fly or maggot classes depending on MODE.
# Output is saved to AUG_DIR and only needs to be run once per MODE.
# =============================================================================

def apply_blue_tint(pil_image, intensity=BLUE_TINT_INTENSITY):
    """
    Boosts the blue channel and reduces red/green.
    intensity: 0.0 = no effect, 1.0 = full blue.
    """
    img_np = np.array(pil_image).astype(float)
    img_np[:, :, 0] = img_np[:, :, 0] * (1 - intensity)                        # red   ↓
    img_np[:, :, 1] = img_np[:, :, 1] * (1 - intensity)                        # green ↓
    img_np[:, :, 2] = np.clip(img_np[:, :, 2] * (1 + intensity * 0.5), 0, 255) # blue  ↑
    return Image.fromarray(img_np.astype(np.uint8))


def apply_cutout(pil_image, cutout_size=CUTOUT_SIZE):
    """
    Applies a single square zero-mask at a random location.
    Patch centre sampled uniformly; may extend outside borders.
    """
    img_np = np.array(pil_image).copy()
    h, w   = img_np.shape[:2]
    cx     = random.randint(0, w - 1)
    cy     = random.randint(0, h - 1)
    x1, x2 = max(0, cx - cutout_size // 2), min(w, cx + cutout_size // 2)
    y1, y2 = max(0, cy - cutout_size // 2), min(h, cy + cutout_size // 2)
    img_np[y1:y2, x1:x2] = 0
    return Image.fromarray(img_np)


# --- Filter classes by MODE ---
eligible_classes = defaultdict(list)
for path, label in dataset.samples:
    class_name = dataset.classes[label].lower()
    if MODE == 'maggots' and 'maggot' in class_name:
        eligible_classes[label].append((path, label))
    elif MODE == 'flies' and 'maggot' not in class_name:
        eligible_classes[label].append((path, label))

# --- Save originals + augmentations to AUG_DIR ---
total_saved = 0
for label, samples in sorted(eligible_classes.items()):
    class_name = dataset.classes[label]
    out_dir    = os.path.join(AUG_DIR, class_name)
    os.makedirs(out_dir, exist_ok=True)

    for path, _ in tqdm(samples, desc=class_name):
        img = Image.open(path).convert('RGB')
        img = img.resize((512, 512), Image.LANCZOS)
        img.save(os.path.join(out_dir, os.path.basename(path)))
        total_saved += 1

        for i, aug_fn in enumerate(AUGMENTATIONS):
            aug_img = aug_fn(img)
            fname   = f"{os.path.splitext(os.path.basename(path))[0]}_aug{i+1}.jpg"
            aug_img.save(os.path.join(out_dir, fname))
            total_saved += 1

print(f"Done. {total_saved} images saved to: {AUG_DIR}")

In [ ]:
# =============================================================================
# CELL 8 — Build trial dataset
# =============================================================================
# Loads the augmented dataset and builds a balanced trial dataset by:
# 1. Filtering classes by MODE and MIN_IMAGES
# 2. Sampling TRIAL_IMAGES_PER_CLASS originals per class
# 3. Including all augmented versions of the sampled originals

# --- Load augmented dataset ---
aug_dataset = SafeImageFolder(AUG_DIR, transform=preprocess, verify=False)
print(f"Augmented dataset: {len(aug_dataset)} images across {len(aug_dataset.classes)} classes")

# --- Separate originals and augmentations by filename ---
originals_by_class     = defaultdict(list)
augmentations_by_class = defaultdict(list)

for path, label in aug_dataset.samples:
    if '_aug' in os.path.basename(path):
        augmentations_by_class[label].append((path, label))
    else:
        originals_by_class[label].append((path, label))

# --- Filter classes by MODE and MIN_IMAGES ---
eligible_classes = {
    label: samples
    for label, samples in originals_by_class.items()
    if len(samples) >= MIN_IMAGES
    and (
        ('maggot' in aug_dataset.classes[label].lower() and MODE == 'maggots')
        or
        ('maggot' not in aug_dataset.classes[label].lower() and MODE == 'flies')
    )
}

print(f"\nClasses included: {len(eligible_classes)}")
for label in sorted(eligible_classes.keys()):
    print(f"  [{label:2d}] {aug_dataset.classes[label]:45s} originals: {len(originals_by_class[label])}  augmented: {len(augmentations_by_class[label])}")

# --- Sample TRIAL_IMAGES_PER_CLASS originals per class ---
# and include all their corresponding augmentations
random.seed(42)
trial_samples = []

for label in sorted(eligible_classes.keys()):
    sampled_originals = random.sample(originals_by_class[label], TRIAL_IMAGES_PER_CLASS)
    trial_samples.extend(sampled_originals)

    sampled_stems = {os.path.splitext(os.path.basename(p))[0] for p, _ in sampled_originals}
    for path, lbl in augmentations_by_class[label]:
        stem = os.path.splitext(os.path.basename(path))[0]
        for i in range(1, len(AUGMENTATIONS) + 1):
            stem = stem.replace(f'_aug{i}', '')
        if stem in sampled_stems:
            trial_samples.append((path, lbl))

print(f"\nTrial dataset: {len(trial_samples)} images across {len(eligible_classes)} classes")
print(f"Expected: {TRIAL_IMAGES_PER_CLASS * (len(AUGMENTATIONS) + 1) * len(eligible_classes)} ({TRIAL_IMAGES_PER_CLASS} × {len(AUGMENTATIONS) + 1} × {len(eligible_classes)} classes)")

In [ ]:
# =============================================================================
# CELL 8b — Store unseen classes
# =============================================================================
# Classes excluded from training = potential unseen/unknown classes.
# Mirrors the MODE filter from Cell 8 but excludes eligible classes.

unseen_classes = {
    label: samples
    for label, samples in originals_by_class.items()
    if label not in eligible_classes
    and (
        ('maggot' in aug_dataset.classes[label].lower() and MODE == 'maggots')
        or
        ('maggot' not in aug_dataset.classes[label].lower() and MODE == 'flies')
    )
}

print(f"Unseen classes (excluded from training): {len(unseen_classes)}")
for label in sorted(unseen_classes.keys()):
    print(f"  [{label:2d}] {aug_dataset.classes[label]:45s} originals: {len(originals_by_class[label])}")

# --- Store unseen samples (originals only, no augmentations) ---
unseen_samples = []
for label in sorted(unseen_classes.keys()):
    unseen_samples.extend(originals_by_class[label])

print(f"\nTotal unseen images: {len(unseen_samples)}")

# --- Sanity check — no overlap between trial and unseen ---
trial_labels  = set(label for _, label in trial_samples)
unseen_labels = set(label for _, label in unseen_samples)
overlap       = trial_labels & unseen_labels

if overlap:
    print(f"⚠️  OVERLAP DETECTED in labels: {overlap}")
else:
    print(f"✅ No overlap between training and unseen classes")

In [ ]:
# =============================================================================
# CELL 9 — Train/Val/Test split
# =============================================================================
# Splits the trial dataset 80/10/10 into train, val, and test sets.
# Originals and their augmentations are always kept in the same split
# to prevent data leakage.

random.seed(42)

# --- Group each original and its augmentations together by filename stem ---
groups = defaultdict(list)
for path, label in trial_samples:
    fname = os.path.basename(path)
    stem  = fname
    for i in range(1, len(AUGMENTATIONS) + 1):
        stem = stem.replace(f'_aug{i}', '')
    stem = os.path.splitext(stem)[0]
    groups[(label, stem)].append((path, label))

# --- Split groups 80/10/10 per class ---
train_samples, val_samples, test_samples = [], [], []

by_class = defaultdict(list)
for key, samples in groups.items():
    label = key[0]
    by_class[label].append(samples)

for label, sample_groups in sorted(by_class.items()):
    random.shuffle(sample_groups)
    n         = len(sample_groups)
    train_end = int(0.8 * n)
    val_end   = int(0.9 * n)

    for group in sample_groups[:train_end]:
        train_samples.extend(group)
    for group in sample_groups[train_end:val_end]:
        val_samples.extend(group)
    for group in sample_groups[val_end:]:
        test_samples.extend(group)

print(f"Train: {len(train_samples)} images")
print(f"Val:   {len(val_samples)} images")
print(f"Test:  {len(test_samples)} images")

In [ ]:
# =============================================================================
# CELL 10 — Embedding extraction utility
# =============================================================================
# Extracts L2-normalised image embeddings from the model for a given set of samples.
# Used for both baseline evaluation and post-training evaluation.

class EvalDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img         = self.transform(Image.open(path).convert('RGB'))
        return img, label


def extract_embeddings(samples, transform, model, device):
    dataset = EvalDataset(samples, transform)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    all_embeddings = []
    all_labels     = []

    model.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Extracting embeddings"):
            imgs = imgs.to(device)
            emb  = model.encode_image(imgs)
            emb  = emb / emb.norm(dim=-1, keepdim=True)
            all_embeddings.append(emb.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_embeddings, axis=0), np.concatenate(all_labels, axis=0)


print("extract_embeddings defined.")

In [ ]:
# ====================================================================================
# CELL 11 — Baseline evaluation (frozen BioCLIP + KNN, can be skipped if unnecessary)
# ====================================================================================
# Evaluates BioCLIP performance before fine-tuning using KNN classification
# on L2-normalised embeddings. Produces a classification report, confusion
# matrix, and t-SNE visualisation.

# --- Extract embeddings ---
print("Extracting train embeddings...")
train_embeddings, train_labels = extract_embeddings(train_samples, preprocess, model, device)

print("Extracting test embeddings...")
test_embeddings, test_labels = extract_embeddings(test_samples, preprocess, model, device)

# --- KNN classification ---
print("\nRunning KNN classifier...")
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(train_embeddings, train_labels)
test_preds = knn.predict(test_embeddings)

# --- Accuracy & classification report ---
overall_acc  = np.mean(test_preds == test_labels)
target_names = [aug_dataset.classes[i] for i in sorted(np.unique(test_labels))]
print(f"\nBaseline Accuracy (untrained {VERSION}): {overall_acc*100:.1f}%")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=target_names))

# --- Confusion matrix ---
fig, ax = plt.subplots(figsize=(12, 10))
cm   = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title(f"Confusion Matrix — Baseline {VERSION} (untrained)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"baseline_confusion_matrix_{VERSION}_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

# --- t-SNE ---
print("\nComputing t-SNE...")
all_embeddings = np.concatenate([train_embeddings, test_embeddings], axis=0)
all_labels     = np.concatenate([train_labels, test_labels], axis=0)
all_splits     = np.array(['train'] * len(train_embeddings) + ['test'] * len(test_embeddings))

tsne          = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings)

fig, ax      = plt.subplots(figsize=(14, 10))
colors       = plt.colormaps['tab20'].resampled(len(np.unique(all_labels)))
unique_labels = sorted(np.unique(all_labels))

for i, label in enumerate(unique_labels):
    for split, marker, size in [('train', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels == label) & (all_splits == split)
        ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   color=colors(i), marker=marker, s=size, alpha=0.6,
                   label=aug_dataset.classes[label] if split == 'train' else None)

ax.set_title(f"t-SNE — Baseline {VERSION} (untrained)\n(circles = train, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"baseline_tsne_{VERSION}_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

print("Done.")

In [ ]:
# =============================================================================
# CELL 12 — TripletDataset & DataLoader
# =============================================================================
# For each anchor image, randomly samples a positive (same class)
# and a negative (different class) to form a triplet for training.

class TripletDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

        self.by_class = defaultdict(list)
        for path, label in samples:
            self.by_class[label].append(path)
        self.labels = list(self.by_class.keys())

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        anchor_path, anchor_label = self.samples[idx]

        # Sample positive: same class, different image
        positive_path = anchor_path
        while positive_path == anchor_path:
            positive_path = random.choice(self.by_class[anchor_label])

        # Sample negative: different class
        negative_label = anchor_label
        while negative_label == anchor_label:
            negative_label = random.choice(self.labels)
        negative_path = random.choice(self.by_class[negative_label])

        anchor   = self.transform(Image.open(anchor_path).convert('RGB'))
        positive = self.transform(Image.open(positive_path).convert('RGB'))
        negative = self.transform(Image.open(negative_path).convert('RGB'))

        return anchor, positive, negative, anchor_label


train_dataset = TripletDataset(train_samples, transform=preprocess)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

print(f"Triplet train dataset: {len(train_dataset)} triplets")
print(f"Batches per epoch:     {len(train_loader)}")

In [ ]:
# =============================================================================
# CELL 13 — LoRA setup
# =============================================================================
# Applies Low-Rank Adaptation (LoRA) to BioCLIP's visual encoder.
# All base model parameters are frozen — only the LoRA layers are trained.
# This keeps the number of trainable parameters very small (~1-2% of total).

# --- Freeze all base model parameters ---
for param in model.parameters():
    param.requires_grad = False

# --- Apply LoRA to attention and MLP layers of the visual encoder ---
lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    target_modules = ["out_proj", "c_fc", "c_proj"],
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

In [ ]:
# =============================================================================
# CELL 14 — Triplet loss with semi-hard mining
# =============================================================================
# For each triplet (anchor, positive, negative) there are three possible cases:
#   Easy:      d(a,n) > d(a,p) + margin  → loss = 0, model already separates them well
#   Semi-hard: d(a,n) > d(a,p) but d(a,n) < d(a,p) + margin  → best learning signal
#   Hard:      d(a,n) < d(a,p)  → negative is closer than positive, unstable early in training
# Only semi-hard triplets are back-propagated. Falls back to all non-easy if none are found.

class TripletLossWithMining(torch.nn.Module):
    def __init__(self, margin=TRIPLET_MARGIN):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        d_pos = torch.sum((anchor - positive) ** 2, dim=1)
        d_neg = torch.sum((anchor - negative) ** 2, dim=1)

        easy      = (d_neg > d_pos + self.margin)
        hard      = (d_neg < d_pos)
        semi_hard = (~easy & ~hard)

        losses = torch.clamp(d_pos - d_neg + self.margin, min=0.0)

        if semi_hard.sum() > 0:
            loss = losses[semi_hard].mean()
        else:
            non_easy = ~easy
            loss = losses[non_easy].mean() if non_easy.sum() > 0 else losses.mean()

        batch_size = anchor.shape[0]
        stats = {
            "easy":          easy.sum().item(),
            "semi_hard":     semi_hard.sum().item(),
            "hard":          hard.sum().item(),
            "easy_pct":      100 * easy.sum().item() / batch_size,
            "semi_hard_pct": 100 * semi_hard.sum().item() / batch_size,
            "hard_pct":      100 * hard.sum().item() / batch_size,
        }

        return loss, stats


triplet_loss_fn = TripletLossWithMining()
print(f"Triplet loss initialised with margin={TRIPLET_MARGIN}")

In [ ]:
# =============================================================================
# CELL 15 — Training loop
# =============================================================================
# Trains the LoRA-adapted BioCLIP model using triplet loss.
# Saves the best LoRA adapter checkpoint to Drive based on validation accuracy.
# Note: train_acc is measured on the training set itself (optimistic upper bound).
# Training history is saved to Drive at the end.

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY
)

best_val_acc = 0.0
history = {
    "train_loss":    [],
    "val_loss":      [],
    "train_acc":     [],
    "val_acc":       [],
    "easy_pct":      [],
    "semi_hard_pct": [],
    "hard_pct":      [],
}


def compute_val_loss(samples, model, device):
    """Computes average triplet loss on validation samples."""
    val_dataset = TripletDataset(samples, transform=preprocess)
    val_loader  = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            anchor, positive, negative, _ = batch
            anchor   = anchor.to(device)
            positive = positive.to(device)
            negative = negative.to(device)

            all_imgs = torch.cat([anchor, positive, negative], dim=0)
            all_embs = model.encode_image(all_imgs)
            anchor_emb, positive_emb, negative_emb = all_embs.chunk(3, dim=0)

            anchor_emb   = anchor_emb   / anchor_emb.norm(dim=-1, keepdim=True)
            positive_emb = positive_emb / positive_emb.norm(dim=-1, keepdim=True)
            negative_emb = negative_emb / negative_emb.norm(dim=-1, keepdim=True)

            loss, _ = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)
            total_loss += loss.item()

    return total_loss / len(val_loader)


for epoch in range(EPOCHS):
    model.train()
    epoch_loss  = 0.0
    epoch_stats = {"easy": 0, "semi_hard": 0, "hard": 0}

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        anchor, positive, negative, _ = batch
        anchor   = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        all_imgs = torch.cat([anchor, positive, negative], dim=0)
        all_embs = model.encode_image(all_imgs)
        anchor_emb, positive_emb, negative_emb = all_embs.chunk(3, dim=0)

        anchor_emb   = anchor_emb   / anchor_emb.norm(dim=-1, keepdim=True)
        positive_emb = positive_emb / positive_emb.norm(dim=-1, keepdim=True)
        negative_emb = negative_emb / negative_emb.norm(dim=-1, keepdim=True)

        loss, stats = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        for k in epoch_stats:
            epoch_stats[k] += stats[k]

    # --- Compute metrics ---
    avg_train_loss = epoch_loss / len(train_loader)
    avg_val_loss   = compute_val_loss(val_samples, model, device)
    total_triplets = sum(epoch_stats.values())

    # --- KNN accuracy ---
    train_emb, train_lbl = extract_embeddings(train_samples, preprocess, model, device)
    val_emb,   val_lbl   = extract_embeddings(val_samples,   preprocess, model, device)

    knn = KNeighborsClassifier(n_neighbors=K, metric='cosine')
    knn.fit(train_emb, train_lbl)
    train_acc = np.mean(knn.predict(train_emb) == train_lbl)   # optimistic: fitted and evaluated on same set
    val_acc   = np.mean(knn.predict(val_emb)   == val_lbl)

    # --- Update history ---
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["easy_pct"].append(100 * epoch_stats["easy"] / total_triplets)
    history["semi_hard_pct"].append(100 * epoch_stats["semi_hard"] / total_triplets)
    history["hard_pct"].append(100 * epoch_stats["hard"] / total_triplets)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"  Train Acc:  {train_acc*100:.1f}%  | Val Acc:  {val_acc*100:.1f}%")
    print(f"  Easy: {history['easy_pct'][-1]:.1f}%  Semi-hard: {history['semi_hard_pct'][-1]:.1f}%  Hard: {history['hard_pct'][-1]:.1f}%")

    # --- Save best checkpoint ---
    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"  Saved best checkpoint (val acc: {best_val_acc*100:.1f}%)")

# --- Reload best checkpoint and extract final embeddings ---
print("\nLoading best checkpoint and extracting final embeddings...")
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()

train_embeddings, train_labels = extract_embeddings(train_samples, preprocess, model, device)
val_embeddings,   val_labels   = extract_embeddings(val_samples,   preprocess, model, device)
test_embeddings,  test_labels  = extract_embeddings(test_samples,  preprocess, model, device)

# --- Save training history ---
with open(os.path.join(PLOT_DIR, f"training_history_{VERSION}_{MODE}_test.json"), "w") as f:
    json.dump(history, f)

print(f"\nTraining complete. Best val acc: {best_val_acc*100:.1f}%")
print(f"Train embeddings: {train_embeddings.shape}")
print(f"Val embeddings:   {val_embeddings.shape}")
print(f"Test embeddings:  {test_embeddings.shape}")


In [ ]:
# =============================================================================
# CELL 16 — KNN classification & report (fine-tuned model)
# =============================================================================
# Classifies test embeddings using KNN trained on train embeddings.
# Prints overall accuracy and a full per-class classification report.

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(train_embeddings, train_labels)

test_preds  = knn.predict(test_embeddings)
overall_acc = np.mean(test_preds == test_labels)

print(f"Overall accuracy ({VERSION}, {MODE}): {overall_acc*100:.1f}%")

target_names = [aug_dataset.classes[i] for i in sorted(np.unique(test_labels))]
print(f"\n{VERSION} Classification Report ({MODE}):")
print(classification_report(test_labels, test_preds, target_names=target_names))

In [ ]:
# =============================================================================
# CELL 17 — Confusion matrix (fine-tuned model)
# =============================================================================
fig, ax = plt.subplots(figsize=(14, 12))
cm      = confusion_matrix(test_labels, test_preds)
disp    = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title(f"Confusion Matrix — {VERSION} + LoRA ({MODE})", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"confusion_matrix_{VERSION}_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 18 — t-SNE plot (fine-tuned model)
# =============================================================================
# Visualises train and test embeddings in 2D using t-SNE.
# Circles = train, stars = test.

all_embeddings = np.concatenate([train_embeddings, test_embeddings], axis=0)
all_labels     = np.concatenate([train_labels,     test_labels],     axis=0)
all_splits     = np.array(['train'] * len(train_embeddings) + ['test'] * len(test_embeddings))

print("Computing t-SNE...")
tsne          = TSNE(n_components=2, perplexity=50, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings)

unique_labels = sorted(np.unique(all_labels))
colors        = plt.colormaps['tab20'].resampled(len(unique_labels))

fig, ax = plt.subplots(figsize=(14, 10))
for i, label in enumerate(unique_labels):
    for split, marker, size in [('train', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels == label) & (all_splits == split)
        ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   color=colors(i), marker=marker, s=size, alpha=0.6,
                   label=aug_dataset.classes[label] if split == 'train' else None)

ax.set_title(f"t-SNE — {VERSION} + LoRA ({MODE})\n(circles = train, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
# plt.savefig(os.path.join(PLOT_DIR, f"tsne_plot_2D_{VERSION}_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 19 — UMAP plot (fine-tuned model)
# =============================================================================
# Visualises train and test embeddings in 2D using UMAP.
# Circles = train, stars = test.

print("Computing UMAP...")
reducer = umap.UMAP(
    n_components=2,
    random_state=42,
    n_neighbors=30,
    min_dist=0.1,
    metric="cosine",
)
embeddings_2d_umap = reducer.fit_transform(all_embeddings)

fig, ax = plt.subplots(figsize=(14, 10))
for i, label in enumerate(unique_labels):
    for split, marker, size in [('train', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels == label) & (all_splits == split)
        ax.scatter(
            embeddings_2d_umap[mask, 0],
            embeddings_2d_umap[mask, 1],
            color=colors(i),
            marker=marker,
            s=size,
            alpha=0.6,
            label=aug_dataset.classes[label] if split == 'train' else None,
        )

ax.set_title(f"UMAP — {VERSION} + LoRA ({MODE})\n(circles = train, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"umap_plot_2D_{VERSION}_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 20 — t-SNE plot 3D (fine-tuned model)
# =============================================================================
# Visualises train and test embeddings in 3D using t-SNE (interactive Plotly).
# Circles = train, diamonds = test.

all_embeddings = np.concatenate([train_embeddings, test_embeddings], axis=0)
all_labels     = np.concatenate([train_labels,     test_labels],     axis=0)
all_splits     = np.array(['train'] * len(train_embeddings) + ['test'] * len(test_embeddings))

print("Computing 3D t-SNE...")
tsne          = TSNE(n_components=3, perplexity=50, random_state=42, n_iter=1000)
embeddings_3d = tsne.fit_transform(all_embeddings)

unique_labels = sorted(np.unique(all_labels))
colors_hex    = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
                 '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf',
                 '#aec7e8','#ffbb78']

fig = go.Figure()
for i, label in enumerate(unique_labels):
    color = colors_hex[i % len(colors_hex)]
    name  = aug_dataset.classes[label]

    for split, symbol, size in [('train', 'circle', 4), ('test', 'diamond', 8)]:
        mask = (all_labels == label) & (all_splits == split)
        fig.add_trace(go.Scatter3d(
            x             = embeddings_3d[mask, 0],
            y             = embeddings_3d[mask, 1],
            z             = embeddings_3d[mask, 2],
            mode          = 'markers',
            name          = f"{name} ({split})" if split == 'test' else name,
            legendgroup   = name,
            showlegend    = (split == 'train'),
            marker        = dict(size=size, color=color, symbol=symbol, opacity=0.7),
            hovertemplate = f"<b>{name}</b><br>{split}<extra></extra>",
        ))

fig.update_layout(
    title         = f"3D t-SNE — {VERSION} + LoRA ({MODE})<br><sup>circles = train, diamonds = test</sup>",
    scene         = dict(
        xaxis  = dict(showgrid=False, showticklabels=False, title=''),
        yaxis  = dict(showgrid=False, showticklabels=False, title=''),
        zaxis  = dict(showgrid=False, showticklabels=False, title=''),
        bgcolor = '#111111',
    ),
    paper_bgcolor = '#0e0e0e',
    font          = dict(color='#aaa'),
    legend        = dict(x=1.01, y=1, font=dict(size=9)),
    margin        = dict(l=0, r=0, t=60, b=0),
    width=950, height=700,
)

fig.write_html(os.path.join(PLOT_DIR, f"tsne_3D_{VERSION}_{MODE}.html"))
fig.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 21 — UMAP plot 3D (fine-tuned model)
# =============================================================================
# Visualises train and test embeddings in 3D using UMAP (interactive Plotly).
# Circles = train, diamonds = test.

print("Computing 3D UMAP...")
reducer = umap.UMAP(
    n_components=3,
    random_state=42,
    n_neighbors=30,
    min_dist=0.1,
    metric='cosine',
)
embeddings_3d_umap = reducer.fit_transform(all_embeddings)

fig = go.Figure()
for i, label in enumerate(unique_labels):
    color = colors_hex[i % len(colors_hex)]
    name  = aug_dataset.classes[label]

    for split, symbol, size in [('train', 'circle', 4), ('test', 'diamond', 8)]:
        mask = (all_labels == label) & (all_splits == split)
        fig.add_trace(go.Scatter3d(
            x             = embeddings_3d_umap[mask, 0],
            y             = embeddings_3d_umap[mask, 1],
            z             = embeddings_3d_umap[mask, 2],
            mode          = 'markers',
            name          = f"{name} ({split})" if split == 'test' else name,
            legendgroup   = name,
            showlegend    = (split == 'train'),
            marker        = dict(size=size, color=color, symbol=symbol, opacity=0.7),
            hovertemplate = f"<b>{name}</b><br>{split}<extra></extra>",
        ))

fig.update_layout(
    title         = f"3D UMAP — {VERSION} + LoRA ({MODE})<br><sup>circles = train, diamonds = test</sup>",
    scene         = dict(
        xaxis   = dict(showgrid=False, showticklabels=False, title=''),
        yaxis   = dict(showgrid=False, showticklabels=False, title=''),
        zaxis   = dict(showgrid=False, showticklabels=False, title=''),
        bgcolor = '#111111',
    ),
    paper_bgcolor = '#0e0e0e',
    font          = dict(color='#aaa'),
    legend        = dict(x=1.01, y=1, font=dict(size=9)),
    margin        = dict(l=0, r=0, t=60, b=0),
    width=950, height=700,
)

fig.write_html(os.path.join(PLOT_DIR, f"umap_3D_{VERSION}_{MODE}.html"))
fig.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 22 — ROC curves + Equal Error Rate (EER) (fine-tuned model)
# =============================================================================

# Use the dataset's own mapping — this is the ground truth
idx_to_class = {v: k for k, v in aug_dataset.class_to_idx.items()}

unique_labels    = sorted(np.unique(test_labels))
test_labels_bin  = label_binarize(test_labels, classes=unique_labels)
test_probs       = knn.predict_proba(test_embeddings)

# Guard: ensure KNN classes align with unique_labels
assert list(knn.classes_) == unique_labels, \
    "Mismatch between knn.classes_ and unique_labels — check train/test class overlap."

n_classes = len(unique_labels)
n_cols    = 3
n_rows    = math.ceil(n_classes / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes      = axes.flatten()

eer_results = {}

for i in range(n_classes):
    fpr, tpr, thresholds = roc_curve(test_labels_bin[:, i], test_probs[:, i])
    roc_auc              = auc(fpr, tpr)

    fnr     = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer     = (fpr[eer_idx] + fnr[eer_idx]) / 2

    actual_label = int(knn.classes_[i])
    class_name   = idx_to_class[actual_label]

    eer_results[class_name] = {
        "EER":       round(eer, 4),
        "Threshold": round(float(thresholds[eer_idx]), 4),
        "AUC":       round(roc_auc, 4),
    }

    axes[i].plot(fpr, tpr, lw=2, label=f"AUC={roc_auc:.2f}")
    axes[i].scatter(fpr[eer_idx], tpr[eer_idx], color="red", zorder=5, label=f"EER={eer:.2f}")
    axes[i].plot([0, 1], [1, 0], "k--",  lw=1, alpha=0.4)   # EER line
    axes[i].plot([0, 1], [0, 1], "gray", lw=1, linestyle=":", alpha=0.4)  # random classifier
    axes[i].set_title(f"{class_name}\nAUC={roc_auc:.2f}  EER={eer:.2f}", fontsize=9)
    axes[i].set_xlabel("FPR")
    axes[i].set_ylabel("TPR")
    axes[i].set_xlim([0, 1])
    axes[i].set_ylim([0, 1])
    axes[i].legend(fontsize=7)
    axes[i].grid(True)

for j in range(n_classes, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f"ROC Curves — {VERSION} + LoRA ({MODE}), One-vs-Rest", fontsize=14)
plt.tight_layout()
# plt.savefig(os.path.join(PLOT_DIR, f"roc_curves_{VERSION}_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

# --- EER summary table ---
print(f"\n{'Class':<35} {'EER':>6} {'Threshold':>10} {'AUC':>6}")
print("-" * 60)
for class_name, metrics in eer_results.items():
    print(f"{class_name:<35} {metrics['EER']:>6.4f} {metrics['Threshold']:>10.4f} {metrics['AUC']:>6.4f}")

mean_eer = np.mean([v["EER"] for v in eer_results.values()])
print("-" * 60)
print(f"{'Mean EER':<35} {mean_eer:>6.4f}")

In [ ]:
# =============================================================================
# CELL 23 — KNN distance threshold & unseen rejection (fine-tuned model)
# =============================================================================
# Fits a KNN on training embeddings and sweeps a cosine distance threshold
# to find the best operating point for rejecting unseen (out-of-distribution)
# samples while retaining seen test samples.

# --- unseen_samples already built in Cell 8b — no reload needed ---
print(f"Unseen samples: {len(unseen_samples)} from {len(unseen_classes)} classes")

# --- Extract unseen embeddings (reuses extract_embeddings from Cell 11) ---
print("Extracting unseen embeddings...")
unseen_embeddings, unseen_labels = extract_embeddings(unseen_samples, preprocess, model, device)
print(f"Unseen embeddings: {unseen_embeddings.shape}")

# --- Balance test and unseen to equal size ---
n_eval = min(len(test_embeddings), len(unseen_embeddings))

np.random.seed(42)
idx_test   = np.random.choice(len(test_embeddings),   size=n_eval, replace=False)
idx_unseen = np.random.choice(len(unseen_embeddings), size=n_eval, replace=False)

test_embeddings_balanced   = test_embeddings[idx_test]
unseen_embeddings_balanced = unseen_embeddings[idx_unseen]

print(f"Balanced: {n_eval} seen test vs {n_eval} unseen")
print(f"  (original test: {len(test_embeddings)}, original unseen: {len(unseen_embeddings)})")

# --- Fit KNN on full training embeddings ---
knn_dist = NearestNeighbors(n_neighbors=K, metric="cosine", algorithm="brute")
knn_dist.fit(train_embeddings)
print(f"KNN fitted on {len(train_embeddings)} training embeddings")

# --- Query KNN ---
test_dists,   _ = knn_dist.kneighbors(test_embeddings_balanced)
unseen_dists, _ = knn_dist.kneighbors(unseen_embeddings_balanced)

test_mean_dist   = test_dists.mean(axis=1)
unseen_mean_dist = unseen_dists.mean(axis=1)

print(f"\nTest   — mean dist: {test_mean_dist.mean():.4f} +/- {test_mean_dist.std():.4f}")
print(f"Unseen — mean dist: {unseen_mean_dist.mean():.4f} +/- {unseen_mean_dist.std():.4f}")

# --- Threshold sweep ---
all_dists = np.concatenate([test_mean_dist, unseen_mean_dist])
all_true  = np.array([1] * n_eval + [0] * n_eval)

thresholds      = np.linspace(all_dists.min(), all_dists.max(), 300)
f1_scores       = []
precisions      = []
recalls         = []
unseen_rejected = []

for thresh in thresholds:
    accepted = all_dists < thresh

    tp = ((accepted)  & (all_true == 1)).sum()
    fp = ((accepted)  & (all_true == 0)).sum()
    tn = ((~accepted) & (all_true == 0)).sum()
    fn = ((~accepted) & (all_true == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    f1_scores.append(f1)
    precisions.append(precision)
    recalls.append(recall)
    unseen_rejected.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

f1_scores       = np.array(f1_scores)
precisions      = np.array(precisions)
recalls         = np.array(recalls)
unseen_rejected = np.array(unseen_rejected)

best_idx    = f1_scores.argmax()
best_thresh = thresholds[best_idx]

print(f"\nBest threshold : {best_thresh:.4f}")
print(f"  F1           : {f1_scores[best_idx]*100:.1f}%")
print(f"  Precision    : {precisions[best_idx]*100:.1f}%")
print(f"  Recall (seen): {recalls[best_idx]*100:.1f}%")
print(f"  Unseen rej.  : {unseen_rejected[best_idx]*100:.1f}%")

# --- Plots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(thresholds, f1_scores       * 100, label="F1",               color="purple",    lw=2.5)
ax.plot(thresholds, precisions      * 100, label="Precision",         color="steelblue", lw=1.5, linestyle="--")
ax.plot(thresholds, recalls         * 100, label="Recall (seen)",     color="green",     lw=1.5, linestyle="--")
ax.plot(thresholds, unseen_rejected * 100, label="Unseen rejected %", color="tomato",    lw=1.5, linestyle="--")
ax.axvline(best_thresh, color="black", linestyle=":", lw=1.5, label=f"Best ({best_thresh:.3f})")
ax.set_title(f"F1 / Precision / Recall vs Distance Threshold\n(n={n_eval} seen vs {n_eval} unseen)")
ax.set_xlabel("Mean Cosine Distance Threshold")
ax.set_ylabel("%")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(test_mean_dist,   bins=30, alpha=0.6, color="steelblue", label=f"Seen test (n={n_eval})")
ax.hist(unseen_mean_dist, bins=30, alpha=0.6, color="tomato",    label=f"Unseen (n={n_eval})")
ax.axvline(best_thresh, color="black", linestyle=":", lw=1.5, label=f"Best ({best_thresh:.3f})")
ax.set_title(f"Distance Distribution: Seen vs Unseen\n(KNN fitted on full training set)")
ax.set_xlabel(f"Mean Cosine Distance to {K} Nearest Neighbours")
ax.set_ylabel("Count")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(f"KNN Threshold Analysis — {VERSION} + LoRA ({MODE})", fontsize=13)
plt.tight_layout()
# plt.savefig(os.path.join(PLOT_DIR, f"knn_threshold_{VERSION}_{MODE}.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 24 — Training history plots (fine-tuned model)
# =============================================================================
# Plots loss, accuracy and triplet mining statistics across epochs.

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Loss ---
axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker='o')
axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   marker='o')
axes[0].set_title("Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Triplet Loss")
axes[0].legend()
axes[0].grid(True)

# --- Accuracy ---
axes[1].plot(epochs, [a * 100 for a in history["train_acc"]], label="Train Acc", marker='o')
axes[1].plot(epochs, [a * 100 for a in history["val_acc"]],   label="Val Acc",   marker='o')
axes[1].set_title("Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True)

# --- Mining statistics ---
axes[2].plot(epochs, history["easy_pct"],      label="Easy",      marker='o')
axes[2].plot(epochs, history["semi_hard_pct"], label="Semi-Hard", marker='o')
axes[2].plot(epochs, history["hard_pct"],      label="Hard",      marker='o')
axes[2].set_title("Triplet Mining Statistics per Epoch")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Percentage (%)")
axes[2].legend()
axes[2].grid(True)

plt.suptitle(f"Training History — {VERSION} + LoRA ({MODE})", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"training_history_{VERSION}_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 25 — Attention roll-out (fine-tuned model)
# =============================================================================
# Saves: per-class figure (original | rollout overlay) + raw .npy heatmap matrices.
# Reuses the model already in memory from Cell 15.

# ============================================================
# STEP 1: Attention rollout function
# ============================================================
def get_attention_rollout(model, image_tensor, device, discard_ratio=0.9):
    attention_store = []
    hooks           = []

    for block in model.visual.transformer.resblocks:
        original_forward = block.attn.forward
        store            = {}

        def make_hook(s, orig):
            def patched(query, key, value, **kwargs):
                kwargs['need_weights']        = True
                kwargs['average_attn_weights'] = False
                out, weights = orig(query, key, value, **kwargs)
                s['weights'] = weights
                return out, weights
            return patched

        block.attn.forward = make_hook(store, original_forward)
        hooks.append((block, original_forward, store))

    model.eval()
    with torch.no_grad():
        _ = model.encode_image(image_tensor.unsqueeze(0).to(device))

    all_attentions = []
    for block, orig_fwd, store in hooks:
        block.attn.forward = orig_fwd
        if 'weights' in store:
            attn = store['weights'].squeeze(0).mean(0).cpu().numpy()
            all_attentions.append(attn)

    if len(all_attentions) == 0:
        raise RuntimeError("No attention weights captured.")

    result = np.eye(all_attentions[0].shape[0])
    for attn in all_attentions:
        flat      = attn.flatten()
        threshold = np.percentile(flat, discard_ratio * 100)
        attn      = np.where(attn >= threshold, attn, 0)
        attn      = attn / (attn.sum(axis=-1, keepdims=True) + 1e-8)
        attn_adj  = 0.5 * attn + 0.5 * np.eye(attn.shape[0])
        result    = attn_adj @ result

    cls_attn    = result[0, 1:]
    num_patches = cls_attn.shape[0]
    grid_size   = int(num_patches ** 0.5)
    attn_map    = cls_attn.reshape(grid_size, grid_size)
    attn_map    = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    return attn_map


# ============================================================
# STEP 2: Select images — N_PER_CLASS per class from test set
# ============================================================
N_PER_CLASS = 5

random.seed(42)
class_sample_map = {}
for path, label in test_samples:
    cls = aug_dataset.classes[label]
    if cls not in class_sample_map:
        class_sample_map[cls] = []
    class_sample_map[cls].append((path, label))

selected = []
for cls, samples in class_sample_map.items():
    chosen = random.sample(samples, min(N_PER_CLASS, len(samples)))
    selected.extend(chosen)

print(f"Selected {len(selected)} images across {len(class_sample_map)} classes")

# ============================================================
# STEP 3: Generate figures + save raw heatmap matrices
# ============================================================
ROLLOUT_DIR = os.path.join(PLOT_DIR, f"attention_rollout_{VERSION}_{MODE}")
HEATMAP_DIR = os.path.join(ROLLOUT_DIR, "heatmap_matrices")
os.makedirs(ROLLOUT_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)

# Group by class
by_class = {}
for path, label in selected:
    cls = aug_dataset.classes[label]
    if cls not in by_class:
        by_class[cls] = []
    by_class[cls].append((path, label))

for cls_name, cls_samples in by_class.items():
    print(f"Processing: {cls_name}")
    n    = len(cls_samples)
    fig, axes = plt.subplots(2, n, figsize=(n * 3, 2 * 3))
    if n == 1:
        axes = axes[:, np.newaxis]

    fig.suptitle(f"{cls_name} — Attention Rollout ({VERSION} + LoRA, {MODE})",
                 fontsize=12, fontweight='bold')

    for col, (path, label_idx) in enumerate(cls_samples):
        orig_img   = Image.open(path).convert("RGB")
        img_tensor = preprocess(orig_img)
        orig_np    = np.array(orig_img.resize((224, 224)))

        attn_map    = get_attention_rollout(model, img_tensor, device)
        attn_resized = cv2.resize(attn_map, (224, 224))

        overlay = np.clip(
            0.5 * orig_np / 255. + 0.5 * plt.cm.jet(attn_resized)[:, :, :3], 0, 1
        )

        # Row 0: original
        axes[0, col].imshow(orig_np)
        axes[0, col].axis('off')
        axes[0, col].set_title(f"img {col + 1}", fontsize=8)

        # Row 1: rollout overlay
        axes[1, col].imshow(overlay)
        axes[1, col].axis('off')

        # Row labels (left column only)
        if col == 0:
            axes[0, col].text(-0.3, 0.5, "Original",
                              transform=axes[0, col].transAxes,
                              fontsize=9, fontweight='bold',
                              va='center', ha='right', clip_on=False)
            axes[1, col].text(-0.3, 0.5, f"After LoRA\n(Rollout)",
                              transform=axes[1, col].transAxes,
                              fontsize=9, fontweight='bold',
                              va='center', ha='right', clip_on=False)

        # Save raw heatmap matrix
        safe_cls = cls_name.replace(" ", "_").replace("/", "_")
        img_stem = os.path.splitext(os.path.basename(path))[0]
        npy_path = os.path.join(HEATMAP_DIR, f"{safe_cls}_img{col+1}_{img_stem}_{VERSION}_{MODE}.npy")
        np.save(npy_path, attn_resized)

    plt.tight_layout()
    plt.subplots_adjust(left=0.18)
    safe_cls = cls_name.replace(" ", "_").replace("/", "_")
    fig_path = os.path.join(ROLLOUT_DIR, f"{safe_cls}_rollout_{VERSION}_{MODE}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fig_path}")

print(f"\nAll figures saved to:         {ROLLOUT_DIR}")
print(f"Raw heatmap matrices saved to: {HEATMAP_DIR}")


In [ ]:
# =============================================================================
# CELL 26 — Save reference data for Streamlit app
# =============================================================================
# All filenames are automatically derived from VERSION and MODE (Cell 3).
# Files are saved to EXPORT_DIR (defined in Cell 3).

os.makedirs(EXPORT_DIR, exist_ok=True)

def export(fname, obj=None, mode="npy"):
    """Save a file locally and copy to EXPORT_DIR."""
    if mode == "npy":
        np.save(fname, obj)
    elif mode == "json":
        with open(fname, "w") as f:
            json.dump(obj, f, indent=2)
    elif mode == "pkl":
        with open(fname, "wb") as f:
            pickle.dump(obj, f)
    shutil.copy(fname, os.path.join(EXPORT_DIR, fname))
    print(f"  Saved: {fname}")

print(f"Exporting to {EXPORT_DIR}...\n")

# --- Embeddings & labels ---
export(f"train_embeddings_{VERSION}_{MODE}.npy", train_embeddings,      mode="npy")
export(f"train_labels_{VERSION}_{MODE}.npy",     np.array(train_labels), mode="npy")
export(f"test_embeddings_{VERSION}_{MODE}.npy",  test_embeddings,        mode="npy")
export(f"test_labels_{VERSION}_{MODE}.npy",      np.array(test_labels),  mode="npy")

# --- Class names ---
export(f"class_names_{VERSION}_{MODE}.json", aug_dataset.classes, mode="json")

# --- KNN classifier ---
export(f"knn_classifier_{VERSION}_{MODE}.pkl", knn, mode="pkl")

# --- Best distance threshold (unseen rejection) ---
export(f"best_thresh_{VERSION}_{MODE}.json", {"best_thresh": float(best_thresh), "K": K}, mode="json")

# --- EER results ---
export(f"eer_results_{VERSION}_{MODE}.json", eer_results, mode="json")

print(f"\nExport complete — {VERSION} ({MODE})")
print(f"  Train embeddings : {train_embeddings.shape}")
print(f"  Test embeddings  : {test_embeddings.shape}")
print(f"  Classes          : {aug_dataset.classes}")
print(f"  Best threshold   : {best_thresh:.4f}")
